# Problem 1: Rock-Paper-Scissors with a Teachable Machine model
**Setup (one time)**
1. Train an *Image Project* with 3 classes (Rock, Paper, Scissors) at https://teachablemachine.withgoogle.com/
2. *Export Model -> Tensorflow -> Keras -> Download my model*. Unzip; put `keras_model.h5` and `labels.txt` in `model/`.
3. `pip install tensorflow pillow numpy matplotlib` (if loading fails on newer TensorFlow also `pip install tf_keras`; handled below).
4. Put a new image in `test_images/` (e.g. `test_images/rock.jpg`) and run the last cell.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

MODEL_FILE = "model/keras_model.h5"
LABELS_FILE = "model/labels.txt"

In [ ]:
def load_model(path):
    try:
        import tensorflow as tf
        return tf.keras.models.load_model(path, compile=False)
    except Exception as e:      # Keras 3 can't read some Teachable Machine layers
        print("tf.keras failed (", type(e).__name__, ") - retrying with tf_keras")
        import tf_keras
        return tf_keras.models.load_model(path, compile=False)

model = load_model(MODEL_FILE)

# labels.txt lines look like "0 Rock" -> keep the name only
with open(LABELS_FILE) as f:
    class_names = [line.strip().split(" ", 1)[1] for line in f if line.strip()]
print(class_names)
model.summary()

## Inference (forward propagation)
Teachable Machine preprocessing: resize + center-crop to 224x224, RGB, pixels scaled to [-1, 1].
`model.predict` runs the forward pass and returns softmax probabilities per class.

In [ ]:
def preprocess(path):
    img = Image.open(path).convert("RGB")
    img = ImageOps.fit(img, (224, 224), Image.Resampling.LANCZOS)
    x = (np.asarray(img, dtype=np.float32) / 127.5) - 1.0
    return img, x[np.newaxis, ...]            # shape (1, 224, 224, 3)

def classify(path):
    img, x = preprocess(path)
    probs = model.predict(x, verbose=0)[0]
    k = int(np.argmax(probs))
    plt.imshow(img); plt.axis("off")
    plt.title(f"{class_names[k]} ({probs[k]:.1%})"); plt.show()
    for name, p in zip(class_names, probs):
        print(f"{name:10s} {p:.4f}")
    print("Prediction:", class_names[k])
    return class_names[k]

classify("test_images/rock.jpg")     # change to your new image